In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
y = train["PitNextLap"].astype(int).values

oof_001 = np.load("/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_001_lgb_strict_raw_baseline.npy")
oof_002 = np.load("/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_002_cat_strict_raw_baseline.npy")
oof_003 = np.load("/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_003_lgb_compound_tyrelife_min.npy")
oof_004 = np.load("/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_004_lgb_lap_race_time_norm_min.npy")
oof_006 = np.load("/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_006_cat_original_prior_min_without_normalized.npy")
oof_006b = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260503_006b_cat_original_prior_no_year_prior_min.npy"
)

def report(name, oof):
    print(name)
    print("  AUC:", roc_auc_score(y, oof))
    print()

report("001_lgb_raw", oof_001)
report("002_cat_raw", oof_002)
report("003_lgb_compound_tyrelife", oof_003)
report("004_lgb_lap_race_time_norm", oof_004)
report("006_cat_original_prior_min", oof_006)
report("006b_cat_original_prior_no_year", oof_006b)

pairs = [
    ("002", oof_002, "006", oof_006),
    ("002", oof_002, "006b", oof_006b),
    ("006", oof_006, "006b", oof_006b),
]

for a_name, a, b_name, b in pairs:
    pearson = np.corrcoef(a, b)[0, 1]
    spearman = spearmanr(a, b).correlation
    print(f"{a_name} vs {b_name}: pearson={pearson:.6f}, spearman={spearman:.6f}")

blend_candidates = {
    "avg_002_006": (oof_002 + oof_006) / 2,
    "avg_002_006b": (oof_002 + oof_006b) / 2,
    "avg_006_006b": (oof_006 + oof_006b) / 2,
    "avg_002_006_006b": (oof_002 + oof_006 + oof_006b) / 3,
}

for name, o in blend_candidates.items():
    print(name, roc_auc_score(y, o))

001_lgb_raw
  AUC: 0.940723081019204

002_cat_raw
  AUC: 0.9485020378244504

003_lgb_compound_tyrelife
  AUC: 0.9413109097188121

004_lgb_lap_race_time_norm
  AUC: 0.9416811403305019

006_cat_original_prior_min
  AUC: 0.9487844854601896

006b_cat_original_prior_no_year
  AUC: 0.9487943937294769

002 vs 006: pearson=0.995886, spearman=0.994109
002 vs 006b: pearson=0.995883, spearman=0.994213
006 vs 006b: pearson=0.998881, spearman=0.998705
avg_002_006 0.9489353487187131
avg_002_006b 0.9489403181606149
avg_006_006b 0.94886303918445
avg_002_006_006b 0.9489875349836314
